# Notebook 20 — detached statistical confirmation

Purpose: fixed 56-pair full-JEPA comparison, one GPU, 72-hour original cap. Inputs: the unchanged Notebook 19 optimized kernel and Food101 metadata/TRAIN/DEV; official subset selected and reserved before training, opened only after every final model and TRAIN-fitted classifier freezes. Outputs: artifacts/notebook_20/<RUN_ID>/report.html and final inference, plus full-state checkpoints/checkpoint manifests under checkpoints/<RUN_ID>/.

Expected runtime: approximately 60 hours, capped at72 hours. GPU1 is isolated using CUDA_VISIBLE_DEVICES=1 and logical cuda:0. Full checkpoints retain latest two and best, at epoch boundaries and every50 updates. Exact resume: fresh kernel, sameRUN_ID/code/options/originalclock, Run All in the main Notebook20. Invalid state or IN_FLIGHT time remains UNKNOWN. No optional stopping, candidate switching or fresh test selection.

Detached coordinator and guard run independently of the chat. The guard provides10-second heartbeats and hourly integrity checks. Persistent boot disk deletion policy remains unverified; preserve final/best states off-instance before instance deletion.

In [ ]:
from pathlib import Path
import copy,json,os,shlex,signal,subprocess,time,traceback,shutil,html,base64,io
import nbformat
_SELF=Path(globals().get('__file__','/home/benjamin/workspace/ahr_go/fmm_jeppa/experiments/20_multiday_confirmation/workflow.ipynb'))
_ns={'__file__':str(_SELF.with_name('common.ipynb'))}
for cell in nbformat.read(_SELF.with_name('common.ipynb'),4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,str(_SELF),'exec'),_ns)
C=type('Common',(),_ns)
MAIN='notebooks/20_Multiday_Statistical_Confirmation.ipynb'
MAXIMUM_SECONDS=72*3600.;TRAINING_SECONDS=64*3600.;DEV_SECONDS=70*3600.
TARGET_DEVICE='cuda:0';PHYSICAL_DEVICE='cuda:1'
OPTIONS={'rank':64,'feature_seed':180064,'local_radius':1,'spatial_mode':'disabled','feature_temperature_min':.25,'feature_temperature_max':4.,'feature_temperature_initial':1.,'gate_initial':.5}
SEEDS=list(range(20262301,20262357));TERMINAL={'COMPLETE','UNKNOWN','FAILED','VALIDATION_COMPLETE'}

def artifact_path(root,run_id):
    if not run_id or any(not(c.isalnum()or c in'-_')for c in run_id):raise ValueError('Safe immutable RUN_ID required')
    return Path(root).resolve()/'artifacts/notebook_20'/run_id

def session_names(run_id):
    prefix='fmm20-'+run_id
    return {key:prefix+suffix for key,suffix in [('coordinator',''),('guard','-guard')]}

def consumer_hash(root):return C.digest([c.source for c in nbformat.read(Path(root)/MAIN,4).cells if c.cell_type=='code'])

def status(artifact,state,phase,**details):
    value={'status':state,'phase':phase,'updated_unix':time.time(),**details}
    C.atomic_json(Path(artifact)/'study_status.json',value);print(state,phase,flush=True);return value

def enforce_one_gpu():
    if os.environ.get('CUDA_VISIBLE_DEVICES')!='1':raise RuntimeError('Fresh kernel must set CUDA_VISIBLE_DEVICES=1 before importing torch')
    import torch
    if not torch.cuda.is_available()or torch.cuda.device_count()!=1:raise RuntimeError('Exactly one visible CUDA GPU required')
    return {'CUDA_VISIBLE_DEVICES':'1','logical_device':'cuda:0','physical_gpu_index':1}

def available(root):
    B=C.load(root,'experiments/17_fixed_kernel/benchmark.ipynb','fmm20_gpu_availability')
    return B._exclusive_gpu(PHYSICAL_DEVICE)

def report(root,artifact,result,training=None):
    artifact=Path(artifact);artifact.mkdir(parents=True,exist_ok=True)
    C.atomic_json(artifact/'result.json',result)
    title='Notebook 20: 56 paired JEPA comparisons'
    body='<h1>'+title+'</h1><p>Status: <b>'+html.escape(result.get('status','UNKNOWN'))+'</b></p>'
    body+='<p>GPU 1 only. One fixed optimized kernel (initial local weight 50%) versus ViT; 30 minutes per model; 56 independent paired seeds. One predeclared official-test comparison, no optional stopping.</p>'
    if result.get('status')=='COMPLETE' and result.get('paired_results'):
        import matplotlib;matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        pairs=result['paired_results'];effects=[p['delta_pp']for p in pairs]
        fig,ax=plt.subplots(figsize=(10,5));ax.scatter(range(1,len(effects)+1),effects,s=26,color='#31709a');ax.axhline(0,color='black',lw=1);ax.axhline(.5,color='#a45917',ls='--',label='Meaningful target: 0.5 pp')
        ax.axhline(result['mean_delta_pp'],color='#31709a',label='Mean paired gain');ax.set(xlabel='Registered training pair',ylabel='Kernel minus ViT top-1 accuracy (percentage points)',title='Official test: all registered paired effects');ax.legend();fig.tight_layout()
        p=artifact/'paired_effects.png';fig.savefig(p,dpi=150);plt.close(fig)
        body+='<img style="max-width:100%" src="data:image/png;base64,'+base64.b64encode(p.read_bytes()).decode()+'">'
        body+='<p>Decision: <b>'+html.escape(str(result.get('decision')))+'</b>. Mean gain '+format(result['mean_delta_pp'],'.3f')+' pp; paired 95% interval '+html.escape(str(result.get('raw_ci95_pp')))+'</p>'
    compact={k:v for k,v in result.items()if k not in('source_hashes','paired_results','rows','models')}
    body+='<pre>'+html.escape(json.dumps(compact,indent=2,default=str))+'</pre>'
    (artifact/'report.html').write_text('<!doctype html><html><head><meta charset="utf-8"><title>'+title+'</title><style>body{font:16px system-ui;max-width:1100px;margin:32px auto;padding:0 20px;line-height:1.5}pre{white-space:pre-wrap;background:#f5f7f9;padding:16px}</style></head><body>'+body+'</body></html>')
    return {'report':str(artifact/'report.html')}

def prepare_launch(root,run_id):
    R=C.runner(root);sources=C.source_hashes(root);a=artifact_path(root,run_id+'-preflight-'+C.digest(sources)[:10]);a.mkdir(parents=True,exist_ok=True)
    p=Path(root)/'artifacts/notebook_19_validation/operator/validation_receipt.json';gate=C.read(p)
    if gate.get('status')!='PASS'or gate.get('actual_cuda')!='PASS'or gate.get('operator_export_sha256')!=C.export_hash(Path(root)/'experiments/19_compute_and_gate/operator.ipynb'):raise ValueError('Verified unchanged optimized operator required')
    result_path=a/'actual_preflight_result.json'
    if result_path.exists():proof=C.read(result_path)
    else:proof=R.actual_preflight(root,a,OPTIONS,256,4,device=TARGET_DEVICE)
    if proof.get('status')!='PASS'or not proof.get('passed')or proof.get('source_hashes')!=sources or len(proof.get('arms',{}))!=2:raise ValueError('Current source-bound two-arm GPU resume proof required')
    backend,_=R.bind(root,OPTIONS)
    for arm in proof['arms'].values():
        if arm.get('status')!='PASS'or not arm.get('bitwise_equal')or set(arm.get('checkpoints',{}))!={'reference','resumed'}:raise ValueError('Exact reference and split states required')
        for record in arm['checkpoints'].values():
            d=Path(record['directory']).resolve()
            if not d.is_relative_to(Path(root)/'checkpoints'/a.name):raise ValueError('Proof checkpoint path escaped its run')
            m=validate_manifest_rotation(backend,d)
            if m['successful_updates']!=4 or m['attempted_updates']!=4 or C.file_record(d/m['latest'])['sha256']!=record['sha256']:raise ValueError('Proof checkpoint or step changed')
    pr=proof['plan'];pp=Path(pr['path'])
    if pp!=a/'actual_preflight_plan.json'or C.file_record(pp)['sha256']!=pr['sha256']:raise ValueError('Proof plan identity changed')
    plan=C.read(pp)
    if plan['source_hashes']!=sources or plan['devices']!=[TARGET_DEVICE]or plan['input_workers']!=4:raise ValueError('Proof configuration differs')
    if any(x['config']['training']['microbatch']!=16 or x['config']['training']['accumulation_steps']!=8 or x['config']['image_size']!=256 for x in plan['arms']):raise ValueError('Actual configuration proof required')
    operator={'status':'PASS','passed':True,'source_hashes':sources,'receipt':C.file_record(p),'operator_export_sha256':gate['operator_export_sha256']}
    C.immutable(a/'operator_gate.json',operator)
    return C.file_record(a/'operator_gate.json'),C.file_record(result_path)

def check_registration(root,artifact):
    req=C.read(Path(artifact)/'launch_registration.json');clock=C.read(Path(artifact)/'clock.json')
    if req['sources']!=C.source_hashes(root)or req['consumer_hash']!=consumer_hash(root):raise ValueError('Frozen source or frontdoor changed')
    if clock!=req['clock']or clock['maximum_seconds']!=MAXIMUM_SECONDS or abs(clock['deadline_unix']-clock['started_unix']-MAXIMUM_SECONDS)>1e-5:raise ValueError('Original72h clock changed')
    if req['seeds']!=SEEDS or req['devices']!=[TARGET_DEVICE]or req['physical_gpu']!=1 or req['options']!=OPTIONS or req['active_wall_seconds_per_arm']!=1800:raise ValueError('Registered scientific design changed')
    for key in('operator_gate','preflight'):
        if C.file_record(req[key]['path'])!=req[key]:raise ValueError('Launch proof record changed')
    return req,clock

def gate_receipts(req):
    return [{'role':role,'status':'PASS','passed':True,'evidence':req[key]}for role,key in [('operator','operator_gate'),('exact_resume','preflight')]]

def launch_study(root,run_id='multiday-confirmation-001',profile='full'):
    root=Path(root).resolve();C.bootstrap(root);artifact=artifact_path(root,run_id);artifact.mkdir(parents=True,exist_ok=True)
    if profile=='validation':
        result={'status':'VALIDATION_COMPLETE','fixture_source':'SOFTWARE_ONLY; no training or official pixels','run_id':run_id};report(root,artifact,result);return result
    if profile!='full':raise ValueError('full or validation required')
    enforce_one_gpu()
    from fmm_jepa.food101_training import run_writer
    with run_writer(artifact/'launch_writer'):
        if (artifact/'launch_registration.json').exists():
            req,clock=check_registration(root,artifact)
            saved=C.read(artifact/'study_status.json')if(artifact/'study_status.json').exists()else{}
            if saved.get('status')in TERMINAL:return saved
        else:
            if (artifact/'clock.json').exists():raise ValueError('Orphan clock; preserve for inspection')
            observation=available(root)
            if not observation.get('eligible'):raise RuntimeError('PhysicalGPU1 unavailable')
            operator,proof=prepare_launch(root,run_id)
            observation=available(root)
            if not observation.get('eligible'):raise RuntimeError('PhysicalGPU1 became unavailable')
            storage=C.runner(root).storage_estimate(root,56);free=shutil.disk_usage(root).free
            required=storage.get('conservative_expected_bytes',storage['expected_bytes'])+storage['operational_reserve_bytes']
            if free<required:raise RuntimeError('Insufficient persistent space for fixed56pairs and operational reserve: '+str(storage))
            started=time.time();clock={'started_unix':started,'deadline_unix':started+MAXIMUM_SECONDS,'maximum_seconds':MAXIMUM_SECONDS,'training_deadline_unix':started+TRAINING_SECONDS,'development_deadline_unix':started+DEV_SECONDS,'restart_policy':'Never reset or extend clock'}
            req={'schema':'notebook20-launch/v1','run_id':run_id,'sources':C.source_hashes(root),'consumer_hash':consumer_hash(root),'clock':clock,'options':OPTIONS,'seeds':SEEDS,'devices':[TARGET_DEVICE],'physical_gpu':1,'CUDA_VISIBLE_DEVICES':'1','active_wall_seconds_per_arm':1800.,'input_workers':4,'operator_gate':operator,'preflight':proof,'gpu_admission':observation,'disk_free_bytes_at_launch':free,'storage_estimate':storage,'official_test':'CLOSED_UNTIL_ALL_MODELS_AND_CLASSIFIERS_FROZEN','registered_pair_count':56,'additional_online_parameters':98328}
            C.immutable(artifact/'clock.json',clock);C.immutable(artifact/'launch_registration.json',req)
        C.check_deadline(clock['deadline_unix']);names=session_names(run_id)
        if not(artifact/'report.html').exists():report(root,artifact,{'status':'RUNNING','phase':'registered_launch','run_id':run_id,'clock':clock,'official_test':'CLOSED'})
        for role,function in [('guard','run_guard'),('coordinator','run_coordinator')]:
            path=write_operator_notebook(root,artifact/'operators'/(role+'.ipynb'),function,{'root':str(root),'run_id':run_id});spawn(root,path,names[role],artifact/(role+'.log'))
        result={'status':'DETACHED_PIPELINE_STARTED','run_id':run_id,'sessions':names,'clock':clock,'physical_gpu':1,'report':str(artifact/'report.html'),'attach':'tmux attach -t '+names['coordinator'],'detach':'Ctrl-b then d'}
        C.atomic_json(artifact/'launch_receipt.json',result);return result

def run_coordinator(root,run_id):
    root=Path(root);artifact=artifact_path(root,run_id);req,clock=check_registration(root,artifact);enforce_one_gpu()
    from fmm_jepa.food101_training import run_writer
    monitor=None
    try:
        with run_writer(artifact/'coordinator_writer'):
            R=C.runner(root);F=C.load(root,'experiments/20_multiday_confirmation/confirmation.ipynb','fmm20_confirmation');folder=artifact/'learning';folder.mkdir(parents=True,exist_ok=True)
            p=folder/'plan.json';plan=C.read(p)if p.exists()else R.make_screen_plan(root,run_id,OPTIONS,(TARGET_DEVICE,),clock['deadline_unix'],4,gate_receipts(req))
            C.immutable(p,plan);C.environment(plan['arms'][0]['config']);bundle=R.data_bundle(root,folder/'data',plan['arms'][0]['config'],verify_images=True)
            registration=F.register_confirmation(root,folder,plan,bundle)
            status(artifact,'RUNNING','56_pairs_training',registered_pairs=56,official_test='CLOSED',confirmation_registration=str(folder/'confirmation_registration.json'))
            P=C.load(root,'experiments/19_compute_and_gate/profiling.ipynb','fmm20_gpu_observation');monitor=P.ContinuousGPUObservation(PHYSICAL_DEVICE);monitor.start()
            training=R.train_worker(root,folder,plan,bundle,clock['training_deadline_unix'])
            monitor.stop();foreign=[s for s in monitor.samples if s['pid']not in(None,os.getpid())and(s['sm_percent']or 0)>0]
            qualification={'status':'UNKNOWN_FOREIGN_COMPUTE'if foreign else'OBSERVED_NO_FOREIGN_COMPUTE','own_pid':os.getpid(),'physical_gpu':1,'samples':monitor.samples,'foreign_active_samples':foreign,'scope':'Continuous1Hzobservation; subsecond interference not excluded'};monitor=None
            C.atomic_json(folder/'gpu_performance_qualification.json',qualification)
            if training.get('status')!='COMPLETE'or not training.get('fair_wall_budget_verified'):raise RuntimeError('Incomplete training or wall fairness; official test remains closed')
            if len(training['arms'])!=112:raise ValueError('All112trainedmodels required')
            status(artifact,'RUNNING','final_TRAIN_fitted_DEV_readouts',completed_training_arms=112,official_test='CLOSED')
            evaluation=R.evaluate_worker(root,folder,plan,bundle,training,clock['development_deadline_unix'])
            if evaluation.get('status')!='COMPLETE'or len(evaluation.get('rows',[]))!=112:raise RuntimeError('All112certified final probes required before opening')
            status(artifact,'RUNNING','sealed_official_readout',completed_training_arms=112)
            result=F.seal_and_score(root,folder,plan,bundle,training,{'model_builder':R.builder(root,OPTIONS),'rows':evaluation['rows']},clock['deadline_unix'])
            result.update(run_id=run_id,source_hashes=req['sources'],training_performance_status=qualification['status'],additional_online_parameters=98328,clock=clock,official_test='OPENED_ONLY_AFTER_FROZEN_SEAL')
            report(root,artifact,result,training);status(artifact,result['status'],'finished',decision=result.get('decision'),report=str(artifact/'report.html'));return result
    except BaseException as exc:
        result={'status':'UNKNOWN','reason':repr(exc),'traceback':traceback.format_exc(),'run_id':run_id,'official_test':'CLOSED_UNLESS_VALID_SEAL_EXISTS'}
        report(root,artifact,result);status(artifact,'UNKNOWN','cutoff_or_integrity_failure',reason=repr(exc));return result
    finally:
        if monitor is not None:monitor.stop()

def run_guard(root,run_id):
    root=Path(root);artifact=artifact_path(root,run_id);names=session_names(run_id);last_review=0.;missing_since=None
    try:
        while True:
            req,clock=check_registration(root,artifact);saved=C.read(artifact/'study_status.json')if(artifact/'study_status.json').exists()else{}
            if saved.get('status')in TERMINAL:return {'status':'GUARD_COMPLETE','study_status':saved['status']}
            now=time.time()
            if now>=clock['deadline_unix']:raise TimeoutError('Original72hour deadline exhausted')
            free=shutil.disk_usage(root).free
            if free<2*1024**3:raise RuntimeError('Disk reserve below2GiB; preserve checkpoints and UNKNOWN')
            if not session_exists(names['coordinator']):
                if missing_since is None:missing_since=now
                elif now-missing_since>90:raise RuntimeError('Coordinator disappeared; do not reset timers')
            else:missing_since=None
            progress=[]
            for p in(root/'checkpoints'/run_id).glob('stages/*/arms/*/manifest.json'):
                m=C.read(p);progress.append({'arm':p.parent.name,'successful_updates':m.get('successful_updates'),'attempted_updates':m.get('attempted_updates')})
            C.atomic_json(artifact/'guard_heartbeat.json',{'status':'HEALTHY','updated_unix':now,'deadline_unix':clock['deadline_unix'],'free_bytes':free,'training_progress':progress})
            if now-last_review>=3600:
                backend,_,_=C.modules(root);audits=[]
                for p in(root/'checkpoints'/run_id).glob('stages/*/arms/*/manifest.json'):
                    m=validate_manifest_rotation(backend,p.parent);audits.append({'arm':p.parent.name,'successful_updates':m['successful_updates'],'latest_valid':True})
                C.atomic_json(artifact/'hourly_review.json',{'status':'PASS','updated_unix':now,'audits':audits,'sources_and_clock_valid':True});last_review=now
            time.sleep(10)
    except BaseException as exc:
        stopped=terminate_session_tree(names['coordinator']);result={'status':'UNKNOWN','run_id':run_id,'reason':repr(exc),'stopped_owned_processes':stopped,'official_test':'CLOSED_UNLESS_VALID_SEAL_EXISTS'}
        report(root,artifact,result);status(artifact,'UNKNOWN','guard_failure',reason=repr(exc));return result


def session_exists(name):return subprocess.run(['tmux','has-session','-t','='+name],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL).returncode==0

def terminate_session_tree(session):
    """Stop only processes descending from this exact project-owned tmux session."""
    if not session_exists(session):return []
    roots={int(x)for x in subprocess.check_output(['tmux','list-panes','-t','='+session,'-F','#{pane_pid}'],text=True).split()}
    edges=[tuple(map(int,line.split()))for line in subprocess.check_output(['ps','-eo','pid=,ppid='],text=True).splitlines()];owned=set(roots)
    while True:
        expanded=owned|{pid for pid,parent in edges if parent in owned}
        if expanded==owned:break
        owned=expanded
    for pid in sorted(owned-roots,reverse=True)+sorted(roots):
        try:os.kill(pid,signal.SIGKILL)
        except ProcessLookupError:pass
    subprocess.run(['tmux','kill-session','-t','='+session],capture_output=True);return sorted(owned)

def validate_manifest_rotation(runner,directory):
    for attempt in range(4):
        try:return runner.validate_training_manifest(directory)
        except (FileNotFoundError,ValueError):
            if attempt==3:raise
            time.sleep(.25)

def write_operator_notebook(root,path,function,arguments):
    code="from pathlib import Path\nimport json,nbformat,types,os\nos.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')\nROOT=Path("+repr(str(Path(root).resolve()))+")\nP=ROOT/'experiments/20_multiday_confirmation/workflow.ipynb'\nW=types.ModuleType('fmm20_detached_workflow');W.__file__=str(P)\nfor cell in nbformat.read(P,4).cells:\n    if cell.cell_type=='code'and'ahr-export'in cell.metadata.get('tags',[]):exec(compile(cell.source,str(P),'exec'),W.__dict__)\nRESULT=W."+function+"(**json.loads("+repr(json.dumps(arguments))+"))\nprint(json.dumps(RESULT,indent=2,default=str))\n"
    nb=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Detached20 '+function+'\nGenerated operator entrypoint. Frozen parent source, original deadline and single-writer checkpoints remain authoritative. Resume through Notebook20 with the same RUN_ID.'),nbformat.v4.new_code_cell(code)],metadata={'kernelspec':{'name':'ahr-go','display_name':'AHR FMM-JEPA','language':'python'}})
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        old=nbformat.read(path,4)
        if [c.source for c in old.cells if c.cell_type=='code']!=[c.source for c in nb.cells if c.cell_type=='code']:raise ValueError('Detached entrypoint code changed')
    else:nbformat.write(nb,path)
    return path

def spawn(root,path,session,log):
    if session_exists(session):return {'status':'EXISTING_SESSION','session':session}
    jupyter=Path(root).parent/'.venv/bin/jupyter';command=' '.join(shlex.quote(str(x))for x in[jupyter,'nbconvert','--to','notebook','--execute','--inplace','--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',path])+' >> '+shlex.quote(str(log))+' 2>&1'
    env='CUDA_VISIBLE_DEVICES=1 CUBLAS_WORKSPACE_CONFIG=:4096:8 OMP_NUM_THREADS=2 MKL_NUM_THREADS=2 '
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),env+command],check=True)
    return {'status':'STARTED','session':session,'operator_notebook':str(path),'log':str(log)}

In [ ]:
print('Notebook20 workflow definitions loaded; launch from main Notebook20.')